# AI Agent
It is an intelligent system that receives a high level goal from a user, & autonomously plans, decides, and executes a sequence of actions by using external tools, APIs, or knowledge sources while maintaining context, reasoning over multiple steps, adapting to new information & optimizing for the intended outcome.  

### Charactestics of an AI Agent:
1. Goal Driven: You tell the agent what you want, not how to do it
2. Autonomous Planning: Agent breaks down the problem and sequences tasks on its own
3. Tool Calling: Agent calls APIs, Calculators, Search tools, etc.
4. COntext Aware: Maintains memory across steps to inform future actions
5. Adaptive: Rethinks plan when things change (eg. API Fails, No data found, etc.)

### Coding the AI Agent

In [19]:
import json
import requests
from langchainhub  import Client
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.prompts import PromptTemplate
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_classic.agents import create_react_agent, AgentExecutor

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

# 1. Tools
search_tool = DuckDuckGoSearchRun()

@tool
def get_weather_data(city: str) -> str:
  """This tool fetches the current weather data for a given city"""

  url = f'https://api.weatherstack.com/current?access_key=4d1d8ae207a8c845a52df8a67bf3623e&query={city}'
  response = requests.get(url)

  return response.json()

# 2. Prompts
client = Client()
prompt = client.pull("hwchase17/react")  # pulls the standard ReAct agent prompt
print("Original prompt template:\n", prompt)

prompt_dict = json.loads(prompt)
kwargs = prompt_dict["kwargs"]

prompt = PromptTemplate(
    template=kwargs["template"],
    input_variables=kwargs["input_variables"],
)

print ("Finalized prompt template:\n", prompt)

C:\Users\divya\AppData\Local\Temp\ipykernel_19024\3601886617.py:26: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  prompt = client.pull("hwchase17/react")  # pulls the standard ReAct agent prompt
d:\Repos\Gen-AI\env\Lib\site-packages\langchainhub\client.py:326: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = self.pull_repo(owner_repo_commit)


Original prompt template:
 {"id": ["langchain", "prompts", "prompt", "PromptTemplate"], "lc": 1, "type": "constructor", "kwargs": {"template": "Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}", "input_variables": ["agent_scratchpad", "input", "tool_names", "tools"], "template_format": "f-string", "partial_variables": {}}}
Finalized prompt template:
 input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'] input_types={} partial_variabl

### ReAct Agent:
- It is a design pattern used in AI agents that stands for Reasoning + Acting. 
- It allows a LLM to interleave internal reasoning with external actions (like tool use) in a structured, multi-step process.
- Instead of generating an answer in one go, the model thinks step by step, deciding what it needs to do next and optionally calling tools (APIs, calculators, web search, etc.) to help it.
- ReAct is usefull for:
    1. Multi Step problems
    2. Tool Augmented tasks (web search, database lookup, etc.)
    3. Make the agent's reasoning transparent & auditable
- It was first introduced in a paper "ReAct: SynerGizing Reasoning and Acting in Language Models"

### Agent & Agent Executor:
Agent → the brain (decision maker)  
AgentExecutor → the operator (runs the brain + tools loop)  

1. Agent:
    - It is responsible for deciding what to do next.
    - It uses: LLM, Prompt, Tools
    - It's role is to Think (Thought), Decide (Call a tool or give a final answer)
    - But agent does not execute the tool itself, it only plans the next step

2. Agent Executor:
    - It is the one that actually runs the loop
    - It's job is to ask the agent what's next, Execute tool (if needed), Feed the result back to Agent, Repeat untill final answer

        User Input  
            ↓  
        Agent (decides action)  
            ↓  
        Executor (runs tool)  
            ↓  
        Observation  
            ↓  
        Agent (next step)  
            ↓  
        ... loop continues ...  
            ↓  
        Final Answer  

In [ ]:
# 3. Agent
agent = create_react_agent(
    llm=llm,
    tools=[search_tool, get_weather_data],
    prompt=prompt
)

# 4. Wrapping it with AgentExecutor
agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool, get_weather_data],
    verbose=True
)

# 5. Invoke
response = agent_executor.invoke({"input": "Find the capital of Madhya Pradesh, then find it's current weather condition"})
print(response)



> Entering new AgentExecutor chain...
Thought: To find the capital of Madhya Pradesh, I need to look up the information about the state.
Action: duckduckgo_search
Action Input: capital of Madhya PradeshLess-fertile red-to-yellow soils are spread over much of eastern Madhya Pradesh. ... Madhya Pradesh has a number of national parks and many wildlife ... Madhya Pradesh state is made up of 48 districts, which are grouped into nine divisions: Bhopal , Chambal, Gwalior , Hoshangabad, Indore , Jabalpur ... Madhya Pradesh was created in 1950 from the former British Central Provinces and Berar and the princely states of Makrai and Chhattisgarh, with Nagpur ... ... of Indore, which was once a village at the confluence of the Khan and Saraswati rivers, is the commercial ... Bhopal is the capital of Madhya Pradesh. Madhya Pradesh state is established on 1-Nov-1956 and the capital of Madhya Pradesh is Bhopal. ... Total number of households in Madhya Pradesh State ...Question: Find the capital of

In [21]:
response['output']

'The capital of Madhya Pradesh is Bhopal, and the current weather condition in Bhopal is Sunny with a real-time temperature of 40°C, humidity of 9%, wind of 22.7km/h, pressure of 1004mb, UV of 4.4, and visibility of 10km.'